# 07 — BirdNET, Perch e utilidade preditiva

Audita as entradas dos modelos e reúne macro-F1, acurácia balanceada, matrizes de confusão e distâncias entre embeddings.

> Execute as células de cima para baixo. Parâmetros editáveis aparecem em uma célula própria e todo artefato gerado informa seu destino.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    raise RuntimeError("Execute este notebook dentro do repositório CodEco.")

for location in (ROOT / "src", ROOT / "scripts"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

REPORTS = ROOT / "reports"
DERIVED = ROOT / "data" / "derived"
REPORTS.mkdir(exist_ok=True)
DERIVED.mkdir(exist_ok=True)
print(f"Repositório: {ROOT}")


In [ ]:
RUN_CLASSIFICATION = False
MODELS = ["birdnet", "perch"]
HEAD = "linear"


In [ ]:
from codeco.protocol import adapter_audit
import pandas as pd
pd.Series(adapter_audit(), name="valor").to_frame()


In [ ]:
if RUN_CLASSIFICATION:
    from classify_embeddings import run_classification
    import config as C
    for corpus in C.DATASETS:
        for model in MODELS:
            run_classification(corpus, model, HEAD)
else:
    print("Classificação não executada. O padding do Perch já foi validado acima.")


In [ ]:
import config as C
tables = []
for corpus in C.DATASETS:
    for model in MODELS:
        path = C.OUT_DIR / f"classification_{corpus}_{model}_{HEAD}.csv"
        if path.is_file():
            table = pd.read_csv(path); table["source_file"] = path.name; tables.append(table)
classification = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
if classification.empty:
    print("Nenhum resultado de classificação disponível.")
else:
    classification.to_csv(REPORTS / "foundation_model_summary.csv", index=False)
    display(classification)
